# Search people and entities

Common search recipes: by name, by attribute (optionally value-filtered and within
date windows), acts at certain dates, and full record inspection.

The attribute names used as examples (`naturalidade`, `faculdade`) come from the
tutorial data; discover the attribute types of this project with
`attribute_values` below.

In [ ]:
import timelink
timelink.version

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()

## People by name

In [ ]:
from timelink.pandas import pname_to_df

df = pname_to_df("Andrade", db=tlnb.db)                # exact/like match
df = pname_to_df("Machado", db=tlnb.db, similar=True)  # with variants
df.head()

## People with an attribute

List the existing values of an attribute first, before filtering by value:

In [ ]:
from timelink.pandas import attribute_values

values = attribute_values("naturalidade", db=tlnb.db)
values

In [ ]:
from timelink.pandas import entities_with_attribute

df = entities_with_attribute(
    entity_type="person",
    the_type="naturalidade",
    the_value="Soure",               # optional; SQL wildcards allowed
    show_elements=["name"],
    more_attributes=["faculdade"],   # extra columns to join in
    db=tlnb.db,
)
df.head()

## Attributes within a date window

Add `dates_in=(after, before)`, dates in `AAAAMMDD` form:

In [ ]:
df = entities_with_attribute(
    entity_type="person",
    the_type="naturalidade",
    dates_in=("16000101", "16501231"),
    show_elements=["name"],
    db=tlnb.db,
)
df.head()

## Acts at certain dates

`Act.the_date` is a string in Kleio `AAAAMMDD` format (unknown parts zeroed),
indexed, so range comparisons work as string comparisons:

In [ ]:
from sqlalchemy import select
from timelink.api.models import Act

with tlnb.db.session() as session:
    stmt = (
        select(Act)
        .where(Act.the_type == "baptismo")
        .where(Act.the_date.between("16400101", "16491231"))
    )
    acts = session.execute(stmt).scalars().all()

for act in acts[:10]:
    print(act.id, act.the_date, act.the_type, act.loc, act.ref)

## One person in kleio notation

`entity.to_kleio()` dumps the full record (attributes, relations, group info):


In [ ]:
from timelink.api.models import Person, Attribute
from sqlalchemy import select

with tlnb.db.session() as session:
    stmt = select(Person).join(
        Person.attributes.and_(
            Attribute.the_type == "jesuita-entrada",
            Attribute.the_value.like("Coimbra"),
        )
    )
    for person in session.execute(stmt).scalars().all()[:5]:
        print(person.to_kleio())